# Ejercicios de Structured Streaming: Modos de Salida y Triggers

Este cuaderno desarrolla las pruebas prácticas de comparación entre los modos de salida `complete`, `update` y `append`, así como la configuración de intervalos de disparo mediante `trigger`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

spark = SparkSession \
    .builder \
    .appName("Streaming IABD WordCount") \
    .master("local[2]") \
    .getOrCreate()

lineasDF = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", "9999") \
    .load()

palabrasDF = lineasDF.select(explode(split(lineasDF.value, ' ')).alias('palabra'))
cantidadDF = palabrasDF.groupBy("palabra").count()

## Ejercicio 1: Comparación de los modos de salida

### 1. Modo `complete`

Al enviar primero `Esta es la primera frase` y luego `Enviamos segunda linea primera`, el modo `complete` muestra en cada micro-lote la tabla íntegra con todas las palabras acumuladas desde el inicio de la consulta.

In [2]:
wordCountComplete = cantidadDF.writeStream \
    .format("console") \
    .outputMode("complete") \
    .start()

wordCountComplete.awaitTermination()

-------------------------------------------
Batch: 1
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
|   Esta|    1|
|     es|    1|
|     la|    1|
|primera|    1|
|  frase|    1|
+-------+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+--------+-----+
| palabra|count|
+--------+-----+
|    Esta|    1|
|      es|    1|
|      la|    1|
| primera|    2|
|   frase|    1|
|Enviamos|    1|
| segunda|    1|
|   linea|    1|
+--------+-----+


### 2. Modo `update`

Al cambiar el modo de salida a `update`, reiniciar la consulta y enviar las mismas dos frases, cada micro-lote muestra únicamente los registros que fueron insertados o cuyo conteo cambió en ese lote específico. En el segundo lote solo aparecen `Enviamos`, `segunda`, `linea` con conteo 1 y `primera` actualizado a 2, omitiendo las palabras de la primera frase que no sufrieron cambios.

In [3]:
wordCountUpdate = cantidadDF.writeStream \
    .format("console") \
    .outputMode("update") \
    .start()

wordCountUpdate.awaitTermination()

-------------------------------------------
Batch: 1
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
|   Esta|    1|
|     es|    1|
|     la|    1|
|primera|    1|
|  frase|    1|
+-------+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+--------+-----+
| palabra|count|
+--------+-----+
| primera|    2|
|Enviamos|    1|
| segunda|    1|
|   linea|    1|
+--------+-----+


### 3. Modo `append`

Al configurar el modo de salida en `append` sobre un DataFrame que contiene una agregación sin marca de agua o watermark, Spark impide iniciar la consulta y lanza la excepción `AnalysisException`. Esto sucede porque el modo `append` exige que las filas emitidas sean definitivas e inmutables, pero en un conteo continuo cualquier palabra puede volver a aparecer en el futuro y modificar su total acumulado.

In [4]:
wordCountAppend = cantidadDF.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()

wordCountAppend.awaitTermination()

AnalysisException: Append output mode not supported when there are streaming aggregations on streaming DataFrames/DataSets without watermark;

## Ejercicio 2: Triggers

Al añadir `.trigger(processingTime="1 minute")`, Spark agrupa todos los mensajes recibidos durante esa ventana de un minuto en un único micro-lote. Si se envían las dos frases con pocos segundos de diferencia, la consola no imprime el resultado de inmediato tras cada línea, sino que espera hasta que se cumple el minuto del disparador y procesa ambas frases juntas en un solo lote.

In [5]:
wordCountTrigger = cantidadDF.writeStream \
    .format("console") \
    .outputMode("complete") \
    .trigger(processingTime="1 minute") \
    .start()

wordCountTrigger.awaitTermination()

-------------------------------------------
Batch: 1
-------------------------------------------
+--------+-----+
| palabra|count|
+--------+-----+
|    Esta|    1|
|      es|    1|
|      la|    1|
| primera|    2|
|   frase|    1|
|Enviamos|    1|
| segunda|    1|
|   linea|    1|
+--------+-----+
